# Task #39 — Quyết định cách encode cặp vùng gửi–nhận (Story #7)

Story #6 phát hiện vùng là tín hiệu mạnh nhất, nhưng cặp vùng gửi–nhận (`primary_seller_state` → `customer_state`) có 415 tổ hợp, chỉ 48 tổ hợp có ≥200 đơn — one-hot trực tiếp dễ overfit vì đa số tổ hợp quá thưa mẫu (mục 4.4, tài liệu bàn giao).

Notebook này khảo sát số liệu thật cho 4 phương án encode, làm căn cứ chọn trước khi đưa vào danh sách đặc trưng cuối cùng (Task #40). Dùng Phương án A (loại `is_delayed=NA`) nhất quán với Story #6.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_labeled.csv", low_memory=False)

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["is_delayed"] = df["is_delayed"].astype("boolean")

dfa = df[df["is_delayed"].notna()].copy()
print("Số đơn Phương án A:", len(dfa))

Số đơn Phương án A: 96476


## Phương án A — 2 cột riêng: `customer_state` + `primary_seller_state` (one-hot độc lập)

In [2]:
n_cust_states = dfa["customer_state"].nunique()
n_seller_states = dfa["primary_seller_state"].nunique()
print("Số state khác nhau — customer_state:", n_cust_states)
print("Số state khác nhau — primary_seller_state:", n_seller_states)
print("Số cột one-hot nếu tách riêng 2 cột:", n_cust_states + n_seller_states)

cust_counts = dfa["customer_state"].value_counts()
seller_counts = dfa["primary_seller_state"].value_counts()
print("\ncustomer_state — nhỏ nhất/lớn nhất theo số đơn:")
print(cust_counts.tail(3))
print(cust_counts.head(3))
print("\nprimary_seller_state — nhỏ nhất/lớn nhất theo số đơn:")
print(seller_counts.tail(3))
print(seller_counts.head(3))

cust_rate = dfa.groupby("customer_state", observed=True)["is_delayed"].agg(["size", "mean"])
cust_rate.columns = ["n_orders", "delay_rate"]
cust_rate["delay_rate_pct"] = (cust_rate["delay_rate"] * 100).round(2)
print("\nTỉ lệ trễ theo customer_state — min/max (không lọc cỡ mẫu):")
print(cust_rate.sort_values("delay_rate_pct")[["n_orders", "delay_rate_pct"]].iloc[[0, -1]])

n_cust_small = (cust_counts < 200).sum()
n_seller_small = (seller_counts < 200).sum()
print(f"\nSố state có <200 đơn — customer: {n_cust_small}/{n_cust_states}, seller: {n_seller_small}/{n_seller_states}")

Số state khác nhau — customer_state: 27
Số state khác nhau — primary_seller_state: 22
Số cột one-hot nếu tách riêng 2 cột: 49

customer_state — nhỏ nhất/lớn nhất theo số đơn:
customer_state
AC    80
AP    67
RR    41
Name: count, dtype: int64
customer_state
SP    40495
RJ    12353
MG    11355
Name: count, dtype: int64

primary_seller_state — nhỏ nhất/lớn nhất theo số đơn:
primary_seller_state
SE    9
PA    8
AM    3
Name: count, dtype: int64
primary_seller_state
SP    68393
MG     7662
PR     7442
Name: count, dtype: int64

Tỉ lệ trễ theo customer_state — min/max (không lọc cỡ mẫu):
                n_orders  delay_rate_pct
customer_state                          
RO                   243            2.88
AL                   397           23.93

Số state có <200 đơn — customer: 4/27, seller: 10/22


## Phương án B — Gộp 27 state thành 5 miền địa lý Brazil, rồi lấy cặp miền gửi–nhận (tối đa 25 tổ hợp)

In [3]:
BRAZIL_REGION = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

dfa["customer_region"] = dfa["customer_state"].map(BRAZIL_REGION)
dfa["seller_region"] = dfa["primary_seller_state"].map(BRAZIL_REGION)

print("State chưa map được (nếu có):")
print("customer:", dfa.loc[dfa["customer_region"].isna(), "customer_state"].unique())
print("seller:", dfa.loc[dfa["seller_region"].isna(), "primary_seller_state"].unique())

dfa["region_pair"] = dfa["seller_region"] + " -> " + dfa["customer_region"]
n_region_pairs = dfa["region_pair"].nunique()
print("\nSố tổ hợp cặp miền thực tế xuất hiện:", n_region_pairs, "(tối đa lý thuyết 25)")

region_pair_counts = dfa["region_pair"].value_counts()
print("Số tổ hợp cặp miền có <200 đơn:", (region_pair_counts < 200).sum(), "/", n_region_pairs)

region_pair_rate = dfa.groupby("region_pair", observed=True)["is_delayed"].agg(["size", "mean"])
region_pair_rate.columns = ["n_orders", "delay_rate"]
region_pair_rate["delay_rate_pct"] = (region_pair_rate["delay_rate"] * 100).round(2)
region_pair_rate_sorted = region_pair_rate.sort_values("delay_rate_pct")
print("\nTệ nhất:")
print(region_pair_rate_sorted.tail(3))
print("\nTốt nhất:")
print(region_pair_rate_sorted.head(3))

State chưa map được (nếu có):
customer: <StringArray>
[]
Length: 0, dtype: str
seller: <StringArray>
[]
Length: 0, dtype: str

Số tổ hợp cặp miền thực tế xuất hiện: 23 (tối đa lý thuyết 25)
Số tổ hợp cặp miền có <200 đơn: 9 / 23

Tệ nhất:
                   n_orders  delay_rate  delay_rate_pct
region_pair                                            
Nordeste -> Sul         139    0.151079           15.11
Sul -> Nordeste         839    0.152563           15.26
Norte -> Nordeste         4         0.5            50.0

Tốt nhất:
                         n_orders  delay_rate  delay_rate_pct
region_pair                                                  
Norte -> Sul                    3         0.0             0.0
Centro-Oeste -> Sudeste       891    0.038159            3.82
Sul -> Centro-Oeste           643    0.041991             4.2


## Phương án C — Cặp state gửi–nhận (415 tổ hợp), one-hot trực tiếp hoặc gộp "other" cho tổ hợp thưa mẫu (đối chiếu lại số liệu Story #6)

In [4]:
dfa["state_pair"] = dfa["primary_seller_state"] + " -> " + dfa["customer_state"]
state_pair_counts = dfa["state_pair"].value_counts()
n_state_pairs = len(state_pair_counts)
n_state_pairs_ge200 = (state_pair_counts >= 200).sum()

print("Số tổ hợp state_pair thực tế xuất hiện:", n_state_pairs)
print("Số tổ hợp có >=200 đơn:", n_state_pairs_ge200)
print("Số tổ hợp có <200 đơn (sẽ gộp 'other' nếu dùng phương án gộp):", n_state_pairs - n_state_pairs_ge200)
print("Tỉ lệ đơn nằm trong nhóm 'other' nếu ngưỡng 200:",
      round(dfa.loc[dfa["state_pair"].map(state_pair_counts) < 200].shape[0] / len(dfa) * 100, 2), "%")

Số tổ hợp state_pair thực tế xuất hiện: 410
Số tổ hợp có >=200 đơn: 48
Số tổ hợp có <200 đơn (sẽ gộp 'other' nếu dùng phương án gộp): 362
Tỉ lệ đơn nằm trong nhóm 'other' nếu ngưỡng 200: 8.55 %


## Phương án D — Target encoding có regularization (m-estimate smoothing) trên cặp state gửi–nhận

`smoothed_rate = (n * rate + m * global_rate) / (n + m)` — tổ hợp càng ít mẫu càng kéo về gần tỉ lệ trung bình toàn cục, tránh overfit mà vẫn giữ được thông tin cặp state (không mất chi tiết như gộp miền).

In [5]:
global_rate = dfa["is_delayed"].mean()
m = 200  # trọng số smoothing = ngưỡng cỡ mẫu đã dùng ở Story #6

state_pair_stats = dfa.groupby("state_pair", observed=True)["is_delayed"].agg(["size", "mean"])
state_pair_stats.columns = ["n_orders", "raw_rate"]
state_pair_stats["smoothed_rate"] = (
    state_pair_stats["n_orders"] * state_pair_stats["raw_rate"] + m * global_rate
) / (state_pair_stats["n_orders"] + m)

print("Tỉ lệ trễ trung bình toàn cục (Phương án A):", round(global_rate * 100, 2), "%")
print("\nVí dụ 5 tổ hợp thưa mẫu nhất — raw_rate bị kéo về gần tỉ lệ trung bình sau smoothing:")
sparse_examples = state_pair_stats.sort_values("n_orders").head(5).copy()
sparse_examples["raw_rate_pct"] = (sparse_examples["raw_rate"] * 100).round(2)
sparse_examples["smoothed_rate_pct"] = (sparse_examples["smoothed_rate"] * 100).round(2)
print(sparse_examples[["n_orders", "raw_rate_pct", "smoothed_rate_pct"]])

print("\nVí dụ 3 tổ hợp mẫu lớn nhất — smoothing gần như không đổi:")
dense_examples = state_pair_stats.sort_values("n_orders", ascending=False).head(3).copy()
dense_examples["raw_rate_pct"] = (dense_examples["raw_rate"] * 100).round(2)
dense_examples["smoothed_rate_pct"] = (dense_examples["smoothed_rate"] * 100).round(2)
print(dense_examples[["n_orders", "raw_rate_pct", "smoothed_rate_pct"]])

Tỉ lệ trễ trung bình toàn cục (Phương án A): 8.11 %

Ví dụ 5 tổ hợp thưa mẫu nhất — raw_rate bị kéo về gần tỉ lệ trung bình sau smoothing:
            n_orders  raw_rate_pct  smoothed_rate_pct
state_pair                                           
AM -> AL           1         100.0               8.57
AM -> MA           1         100.0               8.57
AM -> MG           1           0.0               8.07
BA -> AC           1         100.0               8.57
BA -> RR           1           0.0               8.07

Ví dụ 3 tổ hợp mẫu lớn nhất — smoothing gần như không đổi:
            n_orders  raw_rate_pct  smoothed_rate_pct
state_pair                                           
SP -> SP       30727          6.22               6.23
SP -> RJ        8156          15.5              15.32
SP -> MG        7438          6.36               6.41


## Tổng kết số liệu 4 phương án

| Phương án | Số cột/chiều | Cỡ mẫu nhỏ nhất | Nhóm dưới ngưỡng 200 đơn | Biên độ tỉ lệ trễ (nhóm đủ mẫu) | Mất thông tin |
|---|---|---|---|---|---|
| **A** — 2 cột one-hot riêng (`customer_state`, `primary_seller_state`) | 49 cột (27+22) | 3 đơn (seller AM) | 4/27 customer, 10/22 seller | 2,88% (RO) → 23,93% (AL), khớp Story #6 | Không — giữ nguyên tín hiệu từng state, nhưng mất tương tác state×state (VD SP→RJ có thể khác SP→MG dù cùng seller SP) |
| **B** — Cặp 5 miền địa lý (Norte/Nordeste/Centro-Oeste/Sudeste/Sul) | 23 tổ hợp thực tế (≤25 lý thuyết) | 3 đơn (Norte→Sul) | 9/23 | 3,82% → 15,26% (chỉ tính tổ hợp ≥200 đơn) | Có — gộp nhiều state khác biệt (VD AL 23,93% bị trộn chung Nordeste với các state khác) |
| **C** — 410 cặp state, ngưỡng 200 đơn + gộp "other" | 48 cột (nếu one-hot 48 tổ hợp đủ mẫu) + 1 cột "other" | 200 đơn (theo ngưỡng) | 362/410 tổ hợp bị gộp "other" = 8,55% tổng số đơn | (chưa tính — phụ thuộc gộp) | Nhiều — 8,55% đơn bị trộn vào 1 nhóm "other" không đồng nhất (raw_rate từng tổ hợp trong đó dao động 0–100%) |
| **D** — Target encoding smoothing (m=200) trên 410 cặp state | 1 cột số | — (không loại bỏ tổ hợp nào) | Không cần ngưỡng cứng — tổ hợp thưa mẫu tự động kéo về tỉ lệ trung bình 8,11% | VD SP→SP 6,22%→6,23% (gần như giữ nguyên); tổ hợp 1 đơn dao động 0%/100%→8,07–8,57% (kéo mạnh về trung bình) | Không mất tổ hợp nào, nhưng cần fit encoding chỉ trên tập train để tránh rò rỉ dữ liệu (quan trọng cho Story #8) |

## Quyết định

User chọn **Phương án A** — one-hot 2 cột riêng `customer_state` (27 giá trị) và `primary_seller_state` (22 giá trị). Lý do: giữ nguyên tín hiệu mạnh nhất đã tìm thấy ở Story #6 (RO 2,88% → AL 23,93%), đơn giản, dễ giải thích, không cần xử lý regularization/leakage phức tạp thêm ở Story #8. Đây là đặc trưng đưa vào danh sách cuối cùng ở Task #40.